# tstorch Single-Feature Benchmark Report

Loads the newest pytest-benchmark JSON result from `benchmarks/tstorch/results` and compares `tsfresh` against `tstorch` CPU/GPU for features available in both libraries.

In [1]:
from __future__ import annotations

import json
import re
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "packages/tstorch").exists():
            return candidate
    raise FileNotFoundError("Could not find the pybasin repository root")


repo_root = find_repo_root()
results_dir = repo_root / "benchmarks" / "tstorch" / "results"
result_files = sorted(results_dir.glob("*.json"), key=lambda path: path.stat().st_mtime)
if not result_files:
    raise FileNotFoundError(f"No benchmark JSON files found in {results_dir}")

result_path = result_files[-1]
with result_path.open() as file:
    benchmark_json = json.load(file)

print(f"Loaded benchmark result: {result_path.relative_to(repo_root)}")
print(f"Benchmark timestamp: {benchmark_json.get('datetime', '<unknown>')}")

Loaded benchmark result: benchmarks/tstorch/results/single-feature-20260702-105337.json
Benchmark timestamp: 2026-07-02T08:55:41.353386+00:00


In [2]:
PARAM_RE = re.compile(r"\[(?P<backend>tstorch_cpu|tstorch_gpu|tsfresh)-(?P<feature>.+)\]$")

rows: list[dict[str, object]] = []
for benchmark in benchmark_json["benchmarks"]:
    name = benchmark["name"]
    if "test_single_feature_" not in name:
        continue
    match = PARAM_RE.search(name)
    if match is None:
        continue
    rows.append({
        "feature": match.group("feature"),
        "backend": match.group("backend"),
        "mean_ms": benchmark["stats"]["mean"] * 1000.0,
    })

raw = pd.DataFrame(rows)
if raw.empty:
    raise ValueError("No single-feature benchmark rows were found in the JSON file")

pivot = raw.pivot_table(index="feature", columns="backend", values="mean_ms", aggfunc="mean")
pivot = pivot.reindex(columns=["tsfresh", "tstorch_cpu", "tstorch_gpu"])

# Exclude torch-only features by requiring a tsfresh timing.
common = pivot[pivot["tsfresh"].notna()].copy()
common["torch_cpu_xfaster"] = common["tsfresh"] / common["tstorch_cpu"]
common["torch_gpu_xfaster"] = common["tsfresh"] / common["tstorch_gpu"]

report_table = pd.DataFrame({
    "Feature": common.index,
    "Tsfresh (ms)": common["tsfresh"],
    "Torch CPU (ms)": common["tstorch_cpu"],
    "Torch CPU (xfaster)": common["torch_cpu_xfaster"],
    "Torch GPU (ms)": common["tstorch_gpu"],
    "Torch GPU (xfaster)": common["torch_gpu_xfaster"],
}).sort_values("Feature")

cpu_compared = report_table.dropna(subset=["Tsfresh (ms)", "Torch CPU (ms)"])
gpu_compared = report_table.dropna(subset=["Tsfresh (ms)", "Torch GPU (ms)"])

cpu_faster = cpu_compared[cpu_compared["Torch CPU (xfaster)"] > 1.0]
cpu_slower = cpu_compared[cpu_compared["Torch CPU (xfaster)"] < 1.0]
gpu_faster = gpu_compared[gpu_compared["Torch GPU (xfaster)"] > 1.0]
gpu_slower = gpu_compared[gpu_compared["Torch GPU (xfaster)"] < 1.0]

print("Report")
print(f"Features with tsfresh timing: {len(report_table)}")
print(f"Torch CPU faster than tsfresh: {len(cpu_faster)} / {len(cpu_compared)}")
print(f"Torch GPU faster than tsfresh: {len(gpu_faster)} / {len(gpu_compared)}")
print(f"Torch CPU slower than tsfresh: {len(cpu_slower)}")
print(f"Torch GPU slower than tsfresh: {len(gpu_slower)}")

formatters = {
    "Tsfresh (ms)": "{:.3f}",
    "Torch CPU (ms)": "{:.3f}",
    "Torch CPU (xfaster)": "{:.2f}x",
    "Torch GPU (ms)": "{:.3f}",
    "Torch GPU (xfaster)": "{:.2f}x",
}

display(report_table.style.format(formatters, na_rep="-"))

print("Torch CPU slower than tsfresh")
display(
    cpu_slower.sort_values("Torch CPU (xfaster)")[
        ["Feature", "Tsfresh (ms)", "Torch CPU (ms)", "Torch CPU (xfaster)"]
    ].style.format(formatters, na_rep="-")
)

print("Torch GPU slower than tsfresh")
display(
    gpu_slower.sort_values("Torch GPU (xfaster)")[
        ["Feature", "Tsfresh (ms)", "Torch GPU (ms)", "Torch GPU (xfaster)"]
    ].style.format(formatters, na_rep="-")
)

Report
Features with tsfresh timing: 71
Torch CPU faster than tsfresh: 37 / 71
Torch GPU faster than tsfresh: 14 / 71
Torch CPU slower than tsfresh: 34
Torch GPU slower than tsfresh: 57


,Feature,Tsfresh (ms),Torch CPU (ms),Torch CPU (xfaster),Torch GPU (ms),Torch GPU (xfaster)
feature,,,,,,
abs_energy,abs_energy,1.091,2.721,0.40x,4.711,0.23x
absolute_maximum,absolute_maximum,1.100,6.576,0.17x,2.846,0.39x
absolute_sum_of_changes,absolute_sum_of_changes,1.218,10.197,0.12x,3.348,0.36x
agg_autocorrelation,agg_autocorrelation,5.280,23.352,0.23x,6.501,0.81x
agg_linear_trend,agg_linear_trend,165.088,0.362,455.50x,3.852,42.85x
ar_coefficient,ar_coefficient,44.000,20.988,2.10x,11.420,3.85x
augmented_dickey_fuller,augmented_dickey_fuller,1151.689,82.745,13.92x,3.765,305.86x
autocorrelation,autocorrelation,2.127,7.086,0.30x,2.075,1.03x
benford_correlation,benford_correlation,37.785,39.392,0.96x,205.302,0.18x


Torch CPU slower than tsfresh


,Feature,Tsfresh (ms),Torch CPU (ms),Torch CPU (xfaster)
feature,,,,
fft_aggregated,fft_aggregated,1.373,23.427,0.06x
skewness,skewness,1.051,16.108,0.07x
kurtosis,kurtosis,1.308,16.309,0.08x
longest_strike_above_mean,longest_strike_above_mean,3.506,34.769,0.10x
max_langevin_fixed_point,max_langevin_fixed_point,4.473,40.324,0.11x
symmetry_looking,symmetry_looking,2.619,21.953,0.12x
absolute_sum_of_changes,absolute_sum_of_changes,1.218,10.197,0.12x
longest_strike_below_mean,longest_strike_below_mean,3.145,22.993,0.14x
absolute_maximum,absolute_maximum,1.100,6.576,0.17x


Torch GPU slower than tsfresh


,Feature,Tsfresh (ms),Torch GPU (ms),Torch GPU (xfaster)
feature,,,,
has_duplicate,has_duplicate,1.292,73.814,0.02x
index_mass_quantile,index_mass_quantile,1.101,16.795,0.07x
last_location_of_maximum,last_location_of_maximum,0.669,6.631,0.10x
has_duplicate_max,has_duplicate_max,1.752,16.318,0.11x
mean_n_absolute_max,mean_n_absolute_max,1.907,15.442,0.12x
cwt_coefficients,cwt_coefficients,8.465,66.975,0.13x
mean_second_derivative_central,mean_second_derivative_central,0.498,3.839,0.13x
change_quantiles,change_quantiles,11.875,83.406,0.14x
length,length,0.406,2.805,0.14x
